In [1]:
!pip uninstall -y scikeras scikit-learn
!pip install scikit-learn==1.5.2 scikeras==0.13.0

Found existing installation: scikeras 0.13.0
Uninstalling scikeras-0.13.0:
  Successfully uninstalled scikeras-0.13.0
Found existing installation: scikit-learn 1.5.2
Uninstalling scikit-learn-1.5.2:
  Successfully uninstalled scikit-learn-1.5.2
  Using cached scikit_learn-1.5.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (13 kB)
  Using cached scikeras-0.13.0-py3-none-any.whl.metadata (3.1 kB)
Using cached scikit_learn-1.5.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.9 MB)
Using cached scikeras-0.13.0-py3-none-any.whl (26 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.44 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.


In [2]:
!pip install scikeras

In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Conv2D, MaxPooling2D, Flatten
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

from scikeras.wrappers import KerasClassifier

# Task 1: Hyperparameter Optimization of ANN

Dataset: Breast Cancer Dataset

Optimization Techniques:
1. Manual Search
2. RandomizedSearchCV
3. GridSearchCV

In [4]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()

X = data.data
y = data.target

print(X.shape)
print(y.shape)

(569, 30)
(569,)


In [5]:
X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [6]:
def create_model(neurons=32,
                 optimizer='adam',
                 dropout_rate=0.2):

    model=Sequential()

    model.add(Dense(neurons,
                    activation='relu',
                    input_shape=(30,)))

    model.add(Dropout(dropout_rate))

    model.add(Dense(1,
                    activation='sigmoid'))

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [7]:
neurons=[16,32,64]

batch_sizes=[16,32]

epochs_list=[10,20]

best_accuracy=0
best_parameters={}

In [8]:
for neuron in neurons:

    for batch in batch_sizes:

        for epoch in epochs_list:

            print("Testing")

            print("Neurons:",neuron)

            print("Batch:",batch)

            print("Epoch:",epoch)

            model=create_model(neurons=neuron)

            history=model.fit(
                X_train,
                y_train,
                epochs=epoch,
                batch_size=batch,
                verbose=0
            )

            loss,accuracy=model.evaluate(
                X_test,
                y_test,
                verbose=0
            )

            print("Accuracy:",accuracy)

            if accuracy>best_accuracy:

                best_accuracy=accuracy

                best_parameters={
                    "neurons":neuron,
                    "batch_size":batch,
                    "epochs":epoch
                }

print(best_accuracy)

print(best_parameters)

Testing
Neurons: 16
Batch: 16
Epoch: 10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Accuracy: 0.9298245906829834
Testing
Neurons: 16
Batch: 16
Epoch: 20
Accuracy: 0.8947368264198303
Testing
Neurons: 16
Batch: 32
Epoch: 10
Accuracy: 0.4385964870452881
Testing
Neurons: 16
Batch: 32
Epoch: 20
Accuracy: 0.9385964870452881
Testing
Neurons: 32
Batch: 16
Epoch: 10
Accuracy: 0.9385964870452881
Testing
Neurons: 32
Batch: 16
Epoch: 20
Accuracy: 0.9473684430122375
Testing
Neurons: 32
Batch: 32
Epoch: 10
Accuracy: 0.7543859481811523
Testing
Neurons: 32
Batch: 32
Epoch: 20
Accuracy: 0.9473684430122375
Testing
Neurons: 64
Batch: 16
Epoch: 10
Accuracy: 0.9385964870452881
Testing
Neurons: 64
Batch: 16
Epoch: 20
Accuracy: 0.9473684430122375
Testing
Neurons: 64
Batch: 32
Epoch: 10
Accuracy: 0.9385964870452881
Testing
Neurons: 64
Batch: 32
Epoch: 20
Accuracy: 0.9473684430122375
0.9473684430122375
{'neurons': 32, 'batch_size': 16, 'epochs': 20}


In [9]:
model=KerasClassifier(
    model=create_model,
    verbose=0
)

In [10]:
def create_model(neurons=32, optimizer="adam", dropout_rate=0.2):
    model = Sequential()
    model.add(Dense(neurons, activation="relu", input_shape=(30,)))
    model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [11]:
param_dist = {
    "model__neurons": [16, 32, 64],
    "batch_size": [16, 32],
    "epochs": [10, 20],
    "model__optimizer": ["adam", "rmsprop"],
    "model__dropout_rate": [0.2, 0.3]
}

In [12]:
random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=5,
    cv=3,
    random_state=42,
    n_jobs=1
)

random_search.fit(X_train, y_train)

print("Best Parameters:", random_search.best_params_)
print("Best Score:", random_search.best_score_)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr

Best Parameters: {'model__optimizer': 'adam', 'model__neurons': 64, 'model__dropout_rate': 0.2, 'epochs': 20, 'batch_size': 32}
Best Score: 0.9010543743464622


In [13]:
param_grid = {
    "model__neurons": [16, 32],
    "batch_size": [16, 32],
    "epochs": [10, 20],
    "model__optimizer": ["adam", "rmsprop"],
    "model__dropout_rate": [0.2, 0.3]
}

In [14]:
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    n_jobs=1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best Score:", grid_search.best_score_)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr

Best Parameters: {'batch_size': 16, 'epochs': 20, 'model__dropout_rate': 0.3, 'model__neurons': 16, 'model__optimizer': 'rmsprop'}
Best Score: 0.9054984315092366


# Task 2: Hyperparameter Optimization of CNN

Dataset: MNIST Handwritten Digit Dataset

Optimization Techniques:
1. Manual Search
2. RandomizedSearchCV
3. GridSearchCV

In [15]:
from tensorflow.keras.datasets import mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()

print(X_train.shape)
print(X_test.shape)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
(60000, 28, 28)
(10000, 28, 28)


In [16]:
X_train = X_train / 255.0
X_test = X_test / 255.0

# Add channel dimension for CNN
X_train = X_train.reshape(-1,28,28,1)
X_test = X_test.reshape(-1,28,28,1)

print(X_train.shape)

(60000, 28, 28, 1)


In [17]:
def create_cnn(filters=32, optimizer='adam'):

    model = Sequential()

    model.add(
        Conv2D(
            filters,
            (3,3),
            activation='relu',
            input_shape=(28,28,1)
        )
    )

    model.add(MaxPooling2D((2,2)))

    model.add(Flatten())

    model.add(Dense(64, activation='relu'))

    model.add(Dense(10, activation='softmax'))


    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [18]:
filters_list = [16,32]
epochs_list = [2,3]
batch_sizes = [32]

In [19]:
for filters in filters_list:

    for epochs in epochs_list:

        for batch in batch_sizes:

            print("Filters:",filters,
                  "Epochs:",epochs,
                  "Batch:",batch)


            model = create_cnn(filters=filters)

            model.fit(
                X_train,
                y_train,
                epochs=epochs,
                batch_size=batch,
                verbose=0
            )


            loss, accuracy = model.evaluate(
                X_test,
                y_test,
                verbose=0
            )


            print("Accuracy:",accuracy)


            if accuracy > best_accuracy:

                best_accuracy = accuracy

                best_parameters = {
                    "filters":filters,
                    "epochs":epochs,
                    "batch_size":batch
                }


print("Best Accuracy:",best_accuracy)
print("Best Parameters:",best_parameters)

Filters: 16 Epochs: 2 Batch: 32


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Accuracy: 0.9787999987602234
Filters: 16 Epochs: 3 Batch: 32
Accuracy: 0.9764000177383423
Filters: 32 Epochs: 2 Batch: 32
Accuracy: 0.9825999736785889
Filters: 32 Epochs: 3 Batch: 32
Accuracy: 0.9837999939918518
Best Accuracy: 0.9837999939918518
Best Parameters: {'filters': 32, 'epochs': 3, 'batch_size': 32}


In [20]:
cnn_model = KerasClassifier(
    model=create_cnn,
    verbose=0
)

In [21]:
cnn_param_dist = {

    "model__filters": [16, 32],

    "batch_size": [32, 64],

    "epochs": [2, 3],

    "model__optimizer": ["adam", "rmsprop"]
}

In [22]:
cnn_random_search = RandomizedSearchCV(
    estimator=cnn_model,
    param_distributions=cnn_param_dist,
    n_iter=4,
    cv=2,
    random_state=42,
    n_jobs=1
)


cnn_random_search.fit(
    X_train,
    y_train
)


print("Best Parameters:",
      cnn_random_search.best_params_)

print("Best Score:",
      cnn_random_search.best_score_)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regulariz

Best Parameters: {'model__optimizer': 'rmsprop', 'model__filters': 16, 'epochs': 3, 'batch_size': 32}
Best Score: 0.9758


In [23]:
cnn_param_grid = {

    "model__filters": [16, 32],

    "batch_size": [32, 64],

    "epochs": [2, 3],

    "model__optimizer": ["adam", "rmsprop"]
}

In [24]:
cnn_grid_search = GridSearchCV(
    estimator=cnn_model,
    param_grid=cnn_param_grid,
    cv=2,
    n_jobs=1
)


cnn_grid_search.fit(
    X_train,
    y_train
)


print("Best Parameters:",
      cnn_grid_search.best_params_)

print("Best Score:",
      cnn_grid_search.best_score_)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regulariz

Best Parameters: {'batch_size': 32, 'epochs': 3, 'model__filters': 32, 'model__optimizer': 'adam'}
Best Score: 0.9796666666666667
